# Remapeo de identificacion de daños en vigas

_Dependencias correspondientes_

In [26]:
# Librerías para manejo de datos
import pandas as pd
import numpy as np

# Librerías para cálculo de distancias y operaciones espaciales
from scipy.spatial.distance import cdist, euclidean
from scipy.spatial import KDTree

# Librerías para manejo de archivos y directorios
import os
import sys
import glob
from pathlib import Path
import shutil  # Para copiar/mover archivos si necesitas respaldos

# Para navegar directorios relativos desde el notebook
import json  # Por si necesitas leer/escribir configuraciones

# Librerías para visualización (opcional, si necesitas graficar)
import matplotlib.pyplot as plt
import seaborn as sns

# Para copias de seguridad y manejo de datos
import copy
from datetime import datetime

# Para iterar y filtrar de manera eficiente
from itertools import product, combinations
import warnings
warnings.filterwarnings('ignore')  # Para suprimir warnings innecesarios

_Carga de todos_los_resultados_csv: dicho archivo será el que modificaremos._

In [27]:
# Cargar el archivo CSV desde ../jupyter_notebooks/
csv_path = Path('..') / 'jupyter_notebooks' / 'todos_los_resultados_csv.csv'
df = pd.read_csv(csv_path)

print(f"Archivo cargado: {csv_path}")
print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
print("\nPrimeras filas del DataFrame:")
df.head()

Archivo cargado: ../jupyter_notebooks/todos_los_resultados_csv.csv
Dimensiones: 1080 filas x 12 columnas

Primeras filas del DataFrame:


,ID,Elemento,Porcentaje,Tiempo_s,ObjFinal,DeteccionOK,PromDispersion,StdDispersion,MeanAbsDispersion,N_FalsosPositivos,Tipo_elemento_a_buscar,nivel
0,1,1,5,99.483386,0.925560,False,6.411293,7.288737,6.411293,4,Inclined_leg,mudline
1,2,1,10,160.357777,0.909825,False,6.368342,7.668948,6.368342,4,Inclined_leg,mudline
2,3,1,15,225.817945,0.887863,False,6.627271,8.285555,6.627271,4,Inclined_leg,mudline
3,4,1,20,286.486204,0.846761,False,7.599007,9.586079,7.599007,4,Inclined_leg,mudline
4,5,1,25,338.262701,0.796903,False,10.970466,13.557350,10.970466,2,Inclined_leg,mudline


_Copiado del dataframe_

In [28]:
# Crear una copia del DataFrame original para trabajar sin modificar el archivo fuente
df_modificado = df.copy()

print("DataFrame copiado exitosamente.")
print(f"df original: {df.shape}")
print(f"df_modificado: {df_modificado.shape}")
print("\nAhora puedes trabajar con 'df_modificado' sin afectar 'df' ni el archivo original.")

DataFrame copiado exitosamente.
df original: (1080, 12)
df_modificado: (1080, 12)

Ahora puedes trabajar con 'df_modificado' sin afectar 'df' ni el archivo original.


In [29]:
df_modificado.head()

,ID,Elemento,Porcentaje,Tiempo_s,ObjFinal,DeteccionOK,PromDispersion,StdDispersion,MeanAbsDispersion,N_FalsosPositivos,Tipo_elemento_a_buscar,nivel
0,1,1,5,99.483386,0.925560,False,6.411293,7.288737,6.411293,4,Inclined_leg,mudline
1,2,1,10,160.357777,0.909825,False,6.368342,7.668948,6.368342,4,Inclined_leg,mudline
2,3,1,15,225.817945,0.887863,False,6.627271,8.285555,6.627271,4,Inclined_leg,mudline
3,4,1,20,286.486204,0.846761,False,7.599007,9.586079,7.599007,4,Inclined_leg,mudline
4,5,1,25,338.262701,0.796903,False,10.970466,13.557350,10.970466,2,Inclined_leg,mudline


# Conversión de daño parcial a los beam si el nodo con daño identificado se encuentra cercano a la beam

## Bloque 1: Cargar nodos_beams.csv y explorarlo

In [30]:
# Cargar el archivo nodos_beams.csv
nodos_beams_path = Path('.') / 'nodos_beams.csv'
df_nodos_beams = pd.read_csv(nodos_beams_path)

print(f"Archivo cargado: {nodos_beams_path}")
print(f"Dimensiones: {df_nodos_beams.shape[0]} filas x {df_nodos_beams.shape[1]} columnas")
print(f"\nColumnas: {list(df_nodos_beams.columns)}")
print("\nPrimeras filas:")
df_nodos_beams.head(10)

Archivo cargado: nodos_beams.csv
Dimensiones: 16 filas x 5 columnas

Columnas: ['beam', 'nodo_i', 'nodo_j', 'nodo_cercano_1', 'nodo_cercano_2']

Primeras filas:


,beam,nodo_i,nodo_j,nodo_cercano_1,nodo_cercano_2
0,96,36,35,31,39
1,95,34,35,38,30
2,94,33,34,37,29
3,93,33,36,40,32
4,69,28,27,31,24
5,72,26,27,30,23
6,71,25,26,29,22
7,70,25,28,21,32
8,45,17,18,13,22
9,46,18,19,23,14


## Bloque 2: Listar y explorar los CSVs en ./salida_csvs/

In [31]:
# Listar todos los archivos CSV en el directorio salida_csvs
salida_csvs_path = Path('.') / 'salida_csvs'
csv_files = sorted(salida_csvs_path.glob('*.csv'))

print(f"Directorio: {salida_csvs_path}")
print(f"Total de archivos CSV encontrados: {len(csv_files)}")
print("\nPrimeros 10 archivos:")
for i, file in enumerate(csv_files[:10], 1):
    print(f"  {i}. {file.name}")

Directorio: salida_csvs
Total de archivos CSV encontrados: 1080

Primeros 10 archivos:
  1. ID_0001.csv
  2. ID_0002.csv
  3. ID_0003.csv
  4. ID_0004.csv
  5. ID_0005.csv
  6. ID_0006.csv
  7. ID_0007.csv
  8. ID_0008.csv
  9. ID_0009.csv
  10. ID_0010.csv


## Bloque 3: Cargar un CSV de ejemplo y ver su estructura

In [38]:
# Cargar un CSV de ejemplo para ver su estructura
if len(csv_files) > 0:
    ejemplo_csv = csv_files[0]
    df_ejemplo = pd.read_csv(ejemplo_csv)
    
    print(f"Archivo de ejemplo: {ejemplo_csv.name}")
    print(f"Dimensiones: {df_ejemplo.shape}")
    print(f"\nColumnas: {list(df_ejemplo.columns)}")
    print("\nPrimeras 15 filas:")
    print(df_ejemplo.head(15))
    
    # Ver cuántos nodos tienen Estado = "Daño"
    nodos_con_dano = df_ejemplo[df_ejemplo['Estado'] == 'Daño']
    print(f"\n\nNodos detectados con daño: {len(nodos_con_dano)}")
    if len(nodos_con_dano) > 0:
        print("\nNodos con daño:")
        print(nodos_con_dano[['Numero_de_nodo', 'Valor_de_daño_normalizado', 'Estado']])
else:
    print("No se encontraron archivos CSV en salida_csvs/")

Archivo de ejemplo: ID_0001.csv
Dimensiones: (48, 3)

Columnas: ['Numero_de_nodo', 'Valor_de_daño_normalizado', 'Estado']

Primeras 15 filas:
    Numero_de_nodo  Valor_de_daño_normalizado Estado
0                5                  99.988311   Daño
1                6                  80.878867   Daño
2                7                  80.868007   Daño
3                8                 100.000000   Daño
4                9                  55.184013   Daño
5               10                  26.455354      -
6               11                   3.990422      -
7               12                  26.454637      -
8               13                  18.512238      -
9               14                   9.427504      -
10              15                   9.423502      -
11              16                  18.517396      -
12              17                  13.161758      -
13              18                  10.129244      -
14              19                   2.830080      -


Nodos de

## Bloque 4: Función para verificar si un beam tiene detección cercana

In [33]:
def verificar_deteccion_cercana(beam_id, df_nodos_beams, csv_file):
    """
    Verifica si un beam tiene detección en nodos cercanos en un archivo CSV del AG.
    
    Parámetros:
    - beam_id: ID del beam a verificar
    - df_nodos_beams: DataFrame con información de nodos de beams
    - csv_file: Ruta al archivo CSV de resultados del AG
    
    Retorna:
    - True si se detectó daño en nodo_cercano_1 o nodo_cercano_2
    - False en caso contrario
    """
    # Obtener información del beam
    beam_info = df_nodos_beams[df_nodos_beams['beam'] == beam_id]
    
    if beam_info.empty:
        return False
    
    # Obtener los nodos cercanos
    nodo_cercano_1 = beam_info['nodo_cercano_1'].values[0]
    nodo_cercano_2 = beam_info['nodo_cercano_2'].values[0]
    
    # Cargar el CSV del AG
    try:
        df_ag = pd.read_csv(csv_file)
        
        # Filtrar nodos con Estado = "Daño" (con ñ y D mayúscula)
        # Usar strip() para eliminar espacios en blanco
        df_ag['Estado'] = df_ag['Estado'].astype(str).str.strip()
        nodos_con_dano = df_ag[df_ag['Estado'] == 'Daño']['Numero_de_nodo'].values
        
        # Verificar si alguno de los nodos cercanos fue detectado
        if nodo_cercano_1 in nodos_con_dano or nodo_cercano_2 in nodos_con_dano:
            return True
    except Exception as e:
        print(f"Error al procesar {csv_file.name}: {e}")
        return False
    
    return False

# Prueba de la función con un ejemplo
print("Función creada exitosamente (con corrección para 'Daño').")
print("\nPrueba de la función con el primer beam y primer CSV:")
if len(csv_files) > 0 and len(df_nodos_beams) > 0:
    primer_beam = df_nodos_beams['beam'].iloc[0]
    resultado = verificar_deteccion_cercana(primer_beam, df_nodos_beams, csv_files[0])
    print(f"Beam: {primer_beam}")
    print(f"CSV: {csv_files[0].name}")
    print(f"¿Detección cercana?: {resultado}")
    
    # Debug: Verificar qué valores únicos tiene la columna Estado en el CSV
    df_test = pd.read_csv(csv_files[0])
    print(f"\nValores únicos en 'Estado': {df_test['Estado'].unique()}")
    print(f"Nodos con 'Daño': {df_test[df_test['Estado'] == 'Daño']['Numero_de_nodo'].values}")

Función creada exitosamente (con corrección para 'Daño').

Prueba de la función con el primer beam y primer CSV:
Beam: 96
CSV: ID_0001.csv
¿Detección cercana?: False

Valores únicos en 'Estado': ['Daño' '-']
Nodos con 'Daño': [5 6 7 8 9]


## Bloque 5: Aplicar la función a todos los CSVs y actualizar df_modificado

In [34]:
# Aplicar remapeo: cambiar DeteccionOK de False a 0.5 cuando hay detección cercana
print("Iniciando proceso de remapeo...")
print(f"Total de CSVs a procesar: {len(csv_files)}")
print(f"Total de filas en df_modificado: {len(df_modificado)}\n")

# Filtrar solo las filas que corresponden a Beams
filas_beam = df_modificado[df_modificado['Tipo_elemento_a_buscar'] == 'Beam']
print(f"Total de filas con Tipo_elemento_a_buscar = 'Beam': {len(filas_beam)}\n")

# Contador de cambios
cambios_realizados = 0
filas_procesadas = 0

# Iterar sobre cada fila de beams en df_modificado
for idx, row in df_modificado.iterrows():
    # Solo procesar si es un Beam
    if row['Tipo_elemento_a_buscar'] != 'Beam':
        continue
    
    filas_procesadas += 1
    
    # Solo procesar filas donde DeteccionOK es False (0)
    if row['DeteccionOK'] == False or row['DeteccionOK'] == 0:
        
        # Obtener el beam_id de la columna 'ID'
        beam_id = row['ID']
        
        # Para cada CSV, verificar si hay detección cercana
        deteccion_encontrada = False
        for csv_file in csv_files:
            if verificar_deteccion_cercana(beam_id, df_nodos_beams, csv_file):
                deteccion_encontrada = True
                break  # Si encontramos en un CSV, no necesitamos seguir buscando
        
        # Si se encontró detección cercana, cambiar a 0.5
        if deteccion_encontrada:
            df_modificado.at[idx, 'DeteccionOK'] = 0.5
            cambios_realizados += 1
    
    # Mostrar progreso cada 50 filas
    if filas_procesadas % 50 == 0:
        print(f"Procesadas {filas_procesadas} filas de Beams... Cambios realizados: {cambios_realizados}")

print(f"\n✅ Proceso completado!")
print(f"Total de filas de Beams procesadas: {filas_procesadas}")
print(f"Total de cambios realizados (False → 0.5): {cambios_realizados}")

Iniciando proceso de remapeo...
Total de CSVs a procesar: 1080
Total de filas en df_modificado: 1080

Total de filas con Tipo_elemento_a_buscar = 'Beam': 180

Procesadas 50 filas de Beams... Cambios realizados: 0
Procesadas 100 filas de Beams... Cambios realizados: 0
Procesadas 150 filas de Beams... Cambios realizados: 0

✅ Proceso completado!
Total de filas de Beams procesadas: 180
Total de cambios realizados (False → 0.5): 0


## Bloque 6: Verificar cambios y guardar resultado

In [35]:
# Verificar los cambios realizados
print("=== VERIFICACIÓN DE CAMBIOS ===\n")

# Comparar valores de DeteccionOK
print("Distribución de valores en df original:")
print(df['DeteccionOK'].value_counts())

print("\nDistribución de valores en df_modificado:")
print(df_modificado['DeteccionOK'].value_counts())

# Ver algunas filas que cambiaron a 0.5
filas_con_05 = df_modificado[df_modificado['DeteccionOK'] == 0.5]
print(f"\nTotal de filas con DeteccionOK = 0.5: {len(filas_con_05)}")

if len(filas_con_05) > 0:
    print("\nPrimeras 10 filas con DeteccionOK = 0.5:")
    print(filas_con_05.head(10))

# Guardar el DataFrame modificado (opcional, descomenta si quieres guardar)
# output_path = Path('..') / 'jupyter_notebooks' / 'todos_los_resultados_csv_remapeado.csv'
# df_modificado.to_csv(output_path, index=False)
# print(f"\n✅ Archivo guardado en: {output_path}")

=== VERIFICACIÓN DE CAMBIOS ===

Distribución de valores en df original:
DeteccionOK
True     732
False    348
Name: count, dtype: int64

Distribución de valores en df_modificado:
DeteccionOK
True     732
False    348
Name: count, dtype: int64

Total de filas con DeteccionOK = 0.5: 0


## Bloque 7: Convertir True en 1 y False en 0

In [36]:
# Convertir valores booleanos a numéricos en DeteccionOK
# True → 1, False → 0, 0.5 se mantiene como 0.5

print("=== CONVERSIÓN DE BOOLEANOS A NUMÉRICOS ===\n")

print("Valores únicos ANTES de la conversión:")
print(df_modificado['DeteccionOK'].unique())
print(f"\nTipo de datos: {df_modificado['DeteccionOK'].dtype}")

# Convertir True a 1 y False a 0
df_modificado['DeteccionOK'] = df_modificado['DeteccionOK'].replace({True: 1, False: 0})

print("\n" + "="*50)
print("\nValores únicos DESPUÉS de la conversión:")
print(df_modificado['DeteccionOK'].unique())
print(f"\nTipo de datos: {df_modificado['DeteccionOK'].dtype}")

print("\nDistribución final de valores:")
print(df_modificado['DeteccionOK'].value_counts().sort_index())

print("\n✅ Conversión completada:")
print("   • True → 1")
print("   • False → 0")
print("   • 0.5 → 0.5 (detecciones cercanas)")

=== CONVERSIÓN DE BOOLEANOS A NUMÉRICOS ===

Valores únicos ANTES de la conversión:
[False  True]

Tipo de datos: bool


Valores únicos DESPUÉS de la conversión:
[0 1]

Tipo de datos: int64

Distribución final de valores:
DeteccionOK
0    348
1    732
Name: count, dtype: int64

✅ Conversión completada:
   • True → 1
   • False → 0
   • 0.5 → 0.5 (detecciones cercanas)


In [37]:
df_modificado

,ID,Elemento,Porcentaje,Tiempo_s,ObjFinal,DeteccionOK,PromDispersion,StdDispersion,MeanAbsDispersion,N_FalsosPositivos,Tipo_elemento_a_buscar,nivel
0,1,1,5,99.483386,0.925560,0,6.411293,7.288737,6.411293,4,Inclined_leg,mudline
1,2,1,10,160.357777,0.909825,0,6.368342,7.668948,6.368342,4,Inclined_leg,mudline
2,3,1,15,225.817945,0.887863,0,6.627271,8.285555,6.627271,4,Inclined_leg,mudline
3,4,1,20,286.486204,0.846761,0,7.599007,9.586079,7.599007,4,Inclined_leg,mudline
4,5,1,25,338.262701,0.796903,0,10.970466,13.557350,10.970466,2,Inclined_leg,mudline
...,...,...,...,...,...,...,...,...,...,...,...,...
1075,1076,120,25,58185.349031,1.666667,0,20.505603,13.975944,20.505603,13,Beam,splash
1076,1077,120,30,58235.592619,1.666667,0,21.986663,14.074453,21.986663,13,Beam,splash
1077,1078,120,35,58287.232158,1.666667,0,22.935784,14.428474,22.935784,13,Beam,splash
1078,1079,120,40,58335.277747,1.666667,0,21.687879,14.641405,21.687879,13,Beam,splash
